In [1]:
!pip install --upgrade --force-reinstall -q kaggle
!kaggle --version

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.0 which is incompatible.
Kaggle CLI 2.2.4


In [2]:
from google.colab import userdata
import os

token = userdata.get('KAGGLE_API_TOKEN').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token)

print("token file written:", os.path.exists('/root/.kaggle/access_token'))

token file written: True


In [3]:
!kaggle competitions download -c ieee-fraud-detection
!unzip -q ieee-fraud-detection.zip -d data/
!ls data/



sample_submission.csv  test_transaction.csv  train_transaction.csv
test_identity.csv      train_identity.csv


In [4]:
import pandas as pd
import numpy as np

train_tx = pd.read_csv('data/train_transaction.csv')
train_id = pd.read_csv('data/train_identity.csv')

df = train_tx.merge(train_id, on='TransactionID', how='left')
print(df.shape)

(590540, 434)


In [5]:
def downcast(df):
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

df = downcast(df)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")

1681.190136 MB


In [6]:
del train_tx, train_id

In [7]:
fraud_rate = df['isFraud'].mean()
print(f"Fraud rate: {fraud_rate:.4f}")

Fraud rate: 0.0350


In [8]:
link_cols = ['card1','card2','card3','card5','addr1','addr2',
             'P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']
print(df[link_cols].isna().mean().sort_values(ascending=False))

DeviceInfo       0.799055
R_emaildomain    0.767516
DeviceType       0.761557
P_emaildomain    0.159949
addr1            0.111264
addr2            0.111264
card2            0.015127
card5            0.007212
card3            0.002650
card1            0.000000
dtype: float64


In [9]:
print("TransactionDT range:", df['TransactionDT'].min(), "to", df['TransactionDT'].max())
print("Range in days:", (df['TransactionDT'].max() - df['TransactionDT'].min()) / (3600*24))

TransactionDT range: 86400 to 15811131
Range in days: 181.99920138888888


In [10]:
baseline_cols = [
    'TransactionAmt', 'ProductCD', 'card1','card2','card3','card4','card5','card6',
    'addr1','addr2', 'P_emaildomain','R_emaildomain', 'DeviceType',
    'C1','C2','C13','C14',
    'D1','D4','D10',
]

import os
os.makedirs('data', exist_ok=True)
df[baseline_cols + ['isFraud','TransactionDT','TransactionID']].to_parquet('data/baseline_features.parquet')
print("saved.")

saved.
